# FedSwarm — DAY 1: the gate, then A2 (Kaggle GPU)

| step | what | cells | time |
|---|---|---|---|
| 1 | **`gate_fitness`** — which fitness fix closes the degenerate optimum | 8 | ~15 min |
| 2 | **read the verdict and apply it** | — | 1 min |
| 3 | **`a2_reduced`** — does cross-round pheromone memory help at all | 30 | ~6 GPU-h |

**Why A2 first.** FedAWA (CVPR 2025), Adp-FL-PSO and FedPSO already optimise aggregation weights,
and all three are **stateless between rounds**. Cross-round pheromone persistence is the only
structural novelty left, and A2 is the only experiment that tests it.

**Two outcomes that stop you:** step 2 finds no arm that closes the corner (the notebook halts
itself), or step 3 shows `none` ≈ `decayed` ≈ `full` (persistence buys nothing — re-plan before
spending the remaining ~64 GPU-hours). Full context: `HANDOVER.md`.


## How to run this — no uploads, no clicks on data

**Two settings, once** (right sidebar → **Session options**; they stay set for this notebook):

- **Accelerator → GPU T4 x2**
- **Internet → On**

Everything else is automatic: the code is cloned from GitHub and the MRI dataset is attached by
the notebook itself.

**Then, to avoid babysitting it:**

1. Click **Run All** and watch the first few cells (~5 min). You want to see `GPU OK`,
   `INSTALL OK`, and `dataset check: 7200/7200`.
2. Once those pass, **Save Version → Save & Run All (Commit)** and close the tab. The commit
   runs in the background for up to 12 hours and keeps every output. An interactive session can
   die if the browser disconnects; a commit does not.

If anything stops, the error says exactly what to change. The red `ERROR: pip's dependency
resolver…` block during install is **not** one of them — it is harmless, see the install cell.


## 1. Setup — GPU check, then clone from GitHub


In [ ]:
# Settings check FIRST, in two seconds -- not after four minutes of installs. The GPU is the one
# thing no code can switch on; it is a notebook setting, and it persists once set.
import shutil
import subprocess

gpu = (subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
       if shutil.which("nvidia-smi") else None)
if gpu is None or gpu.returncode != 0 or "GPU" not in gpu.stdout:
    raise RuntimeError(
        "NO GPU -- a notebook setting, not a code problem. Fix once and it stays set:\n"
        "  right sidebar -> 'Session options' (or top menu 'Settings') -> Accelerator ->\n"
        "  'GPU T4 x2'. Then Run All again.\n"
        "If the Accelerator menu is greyed out, your Kaggle account needs phone verification\n"
        "(kaggle.com -> Settings -> Phone verification)."
    )
print(gpu.stdout.strip())
print("GPU OK")


In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/researchpaper784-alt/ResearchPaper.git"
REPO_DIR = "/kaggle/working/ResearchPaper"

# BRANCH is not optional. A bare `git clone` takes the DEFAULT branch, `main`, and every config
# and script these notebooks run exists only on the feature branch.
BRANCH = "claude/happy-hamilton-c5jjil"

if not Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

on = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                    capture_output=True, text=True).stdout.strip()
print("branch:", on)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-3"],
                     capture_output=True, text=True).stdout)
if on != BRANCH:
    raise RuntimeError(f"Checked out {on!r}, not {BRANCH!r}.")

# Re-running this cell in a live kernel pulls new code, but Python keeps any fedswarm module it
# already imported -- so a function added since would be "missing". Drop the cached copies.
import importlib  # noqa: E402
import sys  # noqa: E402

stale = [name for name in sys.modules if name == "fedswarm" or name.startswith("fedswarm.")]
for name in stale:
    del sys.modules[name]
importlib.invalidate_caches()
if stale:
    print(f"dropped {len(stale)} cached fedswarm module(s); the fresh code will be imported")


In [ ]:
%cd /kaggle/working/ResearchPaper

# flwr[simulation] pulls in ray; installed first and on its own. fedswarm is --no-deps because
# its own pins target the dev machine and have no Kaggle CUDA build -- Kaggle's base image
# already has a newer working torch/numpy. pip still enforces fedswarm's requires-python
# (>=3.11), so an older runtime fails HERE, loudly, rather than hours later.
#
# EXPECT A RED BLOCK here reading "ERROR: pip's dependency resolver does not currently take into
# account all the packages that are installed", listing bigframes, google-colab, gradio,
# grpcio-tools and others. It is HARMLESS: those are Kaggle's own preinstalled packages
# disagreeing with each other about protobuf/rich/starlette versions, none of which this
# project imports. The line after the installs is the one that matters.
!pip install -q "flwr[simulation]>=1.36.0,<1.37.0"
!pip install -q --no-deps -e .
!pip install -q omegaconf rich
!python -c "import fedswarm, flwr, sys; print('INSTALL OK -- fedswarm importable, flwr', flwr.__version__, '| Python', sys.version.split()[0])"


In [ ]:
import glob
import os
import sys
from pathlib import Path

# The dataset is fetched automatically -- nothing to click. If it is already attached (sidebar,
# or a previous session) that copy is used; otherwise kagglehub attaches the public dataset to
# this session and returns where it mounted.
from fedswarm.data.download import locate_or_fetch_kaggle_dataset  # noqa: E402

print("inputs mounted:", [Path(p).name for p in sorted(glob.glob("/kaggle/input/*"))] or "none yet")
DATA_ROOT = locate_or_fetch_kaggle_dataset()
if DATA_ROOT is None:
    raise RuntimeError(
        "Could not attach masoudnickparvar/brain-tumor-mri-dataset automatically.\n"
        "  Most likely cause: Internet is off. Right sidebar -> 'Session options' -> Internet ->\n"
        "  On, then Run All again. (The clone cell above needs internet too, so if it passed,\n"
        "  this is something else -- attach it by hand once: right sidebar -> '+ Add Input' ->\n"
        "  search 'brain tumor mri dataset' by masoudnickparvar -> (+).)"
    )
DATA_ROOT = str(DATA_ROOT)
print("dataset root:", DATA_ROOT)

# `flwr run` executes an INSTALLED COPY of the app, whose __file__ is not this clone, so the
# app resolves data and cache paths against FEDSWARM_REPO_ROOT rather than its own location.
os.environ["FEDSWARM_DATA_ROOT"] = DATA_ROOT
os.environ["FEDSWARM_REPO_ROOT"] = "/kaggle/working/ResearchPaper"
os.environ["FLWR_DISABLE_RUNTIME_DEPENDENCY_INSTALLATION"] = "1"

sys.path.insert(0, "src")
import torch  # noqa: E402

print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU visible to PyTorch; these sweeps will not finish on CPU. Right sidebar -> "
        "'Session options' -> Accelerator -> 'GPU T4 x2', then Run All again."
    )
print("CPU cores:", os.cpu_count(), "| Python:", sys.version.split()[0])

import flwr  # noqa: E402
from fedswarm.data.download import find_split_parent  # noqa: E402

print("flwr:", flwr.__version__, "| torch:", torch.__version__)
SPLIT_PARENT = find_split_parent(Path(DATA_ROOT))
print("split parent:", SPLIT_PARENT)

# Verify every image is BYTE-IDENTICAL to the one the committed split was built from. A missing
# file would fail loudly later; a re-versioned Kaggle dataset with the same filenames and
# different images would not -- every pseudo-patient boundary would then describe images you
# are not training on, and no result file would show it. ~165 MB of reads, a few seconds.
import hashlib  # noqa: E402

import pandas as pd  # noqa: E402

_manifest = pd.read_csv("data/processed/manifest.csv", dtype={"sha256": str})
missing, changed = [], []
for rel, digest in zip(_manifest["path"], _manifest["sha256"]):
    f = SPLIT_PARENT / rel
    if not f.exists():
        missing.append(rel)
    elif hashlib.sha256(f.read_bytes()).hexdigest() != digest:
        changed.append(rel)
ok = len(_manifest) - len(missing) - len(changed)
print(f"dataset check: {ok}/{len(_manifest)} images present and byte-identical to the manifest")
if missing or changed:
    raise RuntimeError(
        f"{len(missing)} manifest image(s) missing and {len(changed)} with different content "
        f"(first few: {(missing + changed)[:3]}). The attached dataset is not the version the "
        "split was built from -- check you added masoudnickparvar/brain-tumor-mri-dataset and "
        "not a fork, and that Kaggle has not published a new version of it."
    )


### Restore results from a previous session

Skip on your first run. On a re-run, attach this notebook's previous output (**+ Add Input → Your Work**) first.


In [ ]:
# Self-contained on purpose: a Run-All from the middle must not NameError here.
import glob
import shutil
from pathlib import Path

RESULTS = Path("/kaggle/working/ResearchPaper/results")
RESULTS.mkdir(parents=True, exist_ok=True)

restored = 0
for prior in glob.glob("/kaggle/input/**/results", recursive=True):
    for src in Path(prior).rglob("*.json*"):
        dst = RESULTS / src.relative_to(prior)
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
            restored += 1
print(f"restored {restored} file(s) from attached outputs")
for sub in ("gate", "ablation", "main", "robustness"):
    d = RESULTS / "fl" / sub
    print(f"  results/fl/{sub}: {len(list(d.glob('*.json'))) if d.exists() else 0}")


## 2. Build the image cache (~3 min, once per session)


In [ ]:
import pandas as pd

from fedswarm.data.cache import build_and_save_cache
from fedswarm.data.download import find_split_parent, resolve_root

manifest = pd.read_csv("data/processed/manifest.csv")
print("manifest rows:", len(manifest), "| pseudo-patients:", manifest.pseudo_patient_id.nunique())
cache_path = build_and_save_cache(manifest, find_split_parent(resolve_root(None)), 112)
print("cache:", cache_path)


## Federation, GPU, and the gate-fix guard — **not optional**

**GPU.** Ray hides the GPU from any actor requested with `num_gpus=0`, so with no fraction set
every client trains on **CPU** while the server keeps the card — and every logged metric looks
normal. `1/num_clients` lets all ten clients share one T4.

**The gate fix.** Day 1's gate chooses a fitness fix and writes it into `pyproject.toml` —
**inside that Kaggle session only**. It is never pushed to GitHub, so every fresh clone,
including a restart of this notebook, starts on the *broken* default. `ensure_gate_fix()`
re-derives the same patch from the gate's result files (restored from Day 1's saved output) and
re-applies it. It is idempotent, so every cell that runs or reads a FedACO sweep calls it first.

It **raises** instead of printing, because a failing `!` line does not stop Run-All — the
notebook would carry straight on into hours of GPU on the broken objective.


In [ ]:
import subprocess
import sys

GPUS_PER_CLIENT = 0.1   # 1/10 clients


def ensure_gate_fix():
    """Apply the gate's fitness fix to pyproject.toml, or halt Run-All."""
    r = subprocess.run(
        [sys.executable, "scripts/apply_gate_fix.py",
         "--from-results", "results/fl/gate", "--apply-verdict"],
        capture_output=True, text=True,
    )
    print(r.stdout[-3000:])
    if r.stderr.strip():
        print(r.stderr[-1500:])
    if r.returncode != 0:
        raise RuntimeError(
            "No usable fitness fix, so no FedACO sweep may run. Either the gate's results are "
            "not here (attach the Day 1 notebook's output: + Add Input -> Your Work), or the "
            "gate found NO arm that closes the corner -- then re-run the gate with a larger "
            "aco-gamma-entropy before anything else."
        )


print("GPUs per ClientApp:", GPUS_PER_CLIENT)


---
# STEP 1 — the gate: which fitness fix closes the degenerate optimum

8 cells, 120 rounds, **~15 minutes.** On the first real GPU run the best **single-client vertex**
outscored the FedAvg reference point in **15 of 15 rounds** (mean margin **+0.6252**). The colony
did not go there only because the search was too short — so the failure arrives as the search
gets *better*, while every health signal reads normal.

| arm | changes | evidence so far |
|---|---|---|
| `default` | nothing — reproduces the failure | corner won 15/15, +0.6252 |
| `gamma_entropy_0.45` | concentration penalty at its closed-form crossing | a *lower* bound; 0.6 broke everything downstream |
| `dispersion_aggregate` | dispersion shape | local fixture: corner won 0/15, −0.5283 |
| `fedavg` | — | baseline for health check 4 |

**Read `corner_margin`, not macro-F1** — 15 rounds is far too short for macro-F1.


In [ ]:
# The gate must run against the DOCUMENTED defaults, or its `default` arm is not the default.
# If this session already applied a fix, put pyproject back first. Every later cell calls
# ensure_gate_fix(), which re-applies it.
import subprocess
subprocess.run(["git", "checkout", "--", "pyproject.toml"], check=True)

!python scripts/run_sweep_granular.py --config configs/experiment/gate_fitness.yaml --gpus-per-client {GPUS_PER_CLIENT}


In [ ]:
!python scripts/check_fedaco_health.py --results-dir results/fl/gate


---
# STEP 2 — the verdict, and applying it

The rule is **negative in every round, not on average**: the penalty weight and dispersion shape
are set once per run, so a value that clears the mean round leaves the worst rounds degenerate.
The fix goes into **`pyproject.toml`**, the one lever every later sweep inherits — A1 and A2 never
read `configs/strategy/fedaco.yaml`, so patching that file would leave the two sweeps that decide
the paper on the broken default.

**If no arm closed the corner, the next cell halts the notebook.** That is correct: running A2 on
an open corner buys a number that cannot be interpreted.


In [ ]:
import tomllib

ensure_gate_fix()

with open("pyproject.toml", "rb") as fh:
    live = tomllib.load(fh)["tool"]["flwr"]["app"]["config"]
print("live config every later run will use:")
for key in ("aco-gamma-entropy", "aco-dispersion-reference", "aco-q0", "aco-rho-round"):
    print(f"  {key:28} = {live[key]!r}")


---
# STEP 3 — A2: does cross-round pheromone memory do anything?

30 cells, 3,000 rounds, **~6 GPU-h** — one Kaggle session with room. `none`, `decayed`, `full`
across Dirichlet 0.3 and 0.1, 5 seeds. If the session is cut off, re-run the notebook with this
output attached; finished cells are skipped.

5 of these 30 cells are the same resolved config as 5 of Day 2's A1 cells (A2's `decayed` at
Dirichlet 0.3 *is* A1's `aco` there), so Day 2 gets them free.


In [ ]:
ensure_gate_fix()
!python scripts/run_sweep_granular.py --config configs/experiment/ablation_a2_reduced.yaml --gpus-per-client {GPUS_PER_CLIENT}


In [ ]:
# Filter by the run_ids THIS sweep's runner would produce, via the same code path. Other
# sweeps write to the same directory -- A1 and A2 both use results/fl/ablation -- so filtering
# on config values would fold one into the other. ensure_gate_fix() runs before the ids are
# computed, because run_ids hash pyproject's defaults and the fix changes them.
import json
from collections import defaultdict
from pathlib import Path

from fedswarm.sweep import granular_runs, planned_run_ids

ensure_gate_fix()

CONFIG = "configs/experiment/ablation_a2_reduced.yaml"
mine = planned_run_ids(granular_runs(CONFIG, Path(".").resolve()), "pyproject.toml")
print(f"{len(mine)} cells belong to {Path(CONFIG).name}")

f1 = {}   # label -> final test macro-F1
for path in Path("results/fl").rglob("*.json"):
    result = json.loads(path.read_text())
    if result.get("run_id") in mine:
        value = (result.get("final") or {}).get("final_test_macro_f1")
        if value is not None:
            f1[mine[result["run_id"]]] = float(value)
print(f"{len(f1)}/{len(mine)} of this sweep's cells have a result\n")


def mean_std(vals):
    m = sum(vals) / len(vals)
    s = (sum((v - m) ** 2 for v in vals) / (len(vals) - 1)) ** 0.5 if len(vals) > 1 else 0.0
    return m, s


by_arm = defaultdict(list)
for label, value in f1.items():
    arm, partition, _seed = label.split("/")
    by_arm[(arm, partition)].append(value)

if not f1:
    print("Nothing to summarise yet.")
else:
    print(f"{'persistence':12} {'partition':15} {'n':>3} {'mean F1':>9} {'std':>7}")
    for key in sorted(by_arm, key=lambda k: (k[1], k[0])):
        m, s = mean_std(by_arm[key])
        print(f"{key[0]:12} {key[1]:15} {len(by_arm[key]):>3} {m:>9.4f} {s:>7.4f}")
    print("\nIf none / decayed / full sit inside each other's std, cross-round persistence buys")
    print("nothing and the last structural novelty is gone. Stop and re-plan before Day 2.")


In [ ]:
!python scripts/make_tables.py --results-dir results/fl/ablation --out paper/tables


---
# Before the session ends — SAVE, or you lose everything

Kaggle discards `/kaggle/working` unless the notebook is **committed**: use
**Save Version → Save & Run All (Commit)**, not the quick save. Next session, attach this
notebook's output as an input (**+ Add Input → Your Work**) so the restore cell brings it back
and every sweep resumes per-cell.


In [ ]:
from pathlib import Path

results = sorted(Path("results").rglob("*.json"))
print(f"{len(results)} result file(s) to save")
for directory in sorted({p.parent for p in results}):
    print(f"  {directory}: {len(list(directory.glob('*.json')))}")
print("\nSave Version -> Save & Run All (Commit). The quick save does NOT keep /kaggle/working.")
print("NEXT: Day 2 -- notebooks/kaggle_day2_a1.ipynb. Attaching THIS notebook's output saves ~20 min.")
